# 06 — Eksperimen Deep Learning (10 baseline)

10 arsitektur × 5 horizon × 2 setting = **100 run**. Data: harga emas **Antam (IDR)** + 7 indikator makro **stasioner** (`data_gabungan.csv`).

Konfigurasi mengikuti **Table 3 jurnal acuan** (Bao et al., 2026) sedekat mungkin (semua konstanta di `src/config.py`): `SEQ_LEN = 10`; optimizer **AdamW lr = 0,001, weight decay = 0,0001** untuk 10 model baseline ("unified hyperparameters"); z-score fit hanya pada train; kanal eksogen di-clip ±5; MSE; ≤100 epoch; early stopping (sabar 10) dan ReduceLROnPlateau pada **baris validasi ber-embargo**; bobot epoch terbaik dipulihkan.
**DL tidak diubah** dari versi sebelumnya agar tetap sejajar dengan jurnal dan hasilnya sah dipakai ulang.

DPformer (dengan lr/wd hasil MOSSA) ada di **notebook 05**. Uji kestabilan lintas seed ada di **notebook 13**.

**Bobot untuk SHAP:** bobot epoch-terbaik setiap model baseline pada (multivariat, h=20) disimpan di `outputs/results/models/` agar notebook 11 tidak perlu melatih ulang model baseline yang terpilih otomatis. Bila hasil lama dipakai ulang (run dilewati), notebook 11 akan melatih ulang **satu** model terpilih itu dengan seed dan konfigurasi yang sama (dicek: MAE harus sama dengan hasil utama).

**Perkiraan waktu (CPU 2 thread):** ± 2–2,5 jam (BiLSTM/LSTM hidden = 1024 paling lambat). Hasil valid dilewati otomatis.

In [1]:
import sys, os, json, time, warnings

# Batasi thread CPU supaya laptop tetap responsif selama training berjam-jam.
# Default 2 thread. Ubah lewat env TA_THREADS. Harus di kernel BARU (sebelum numpy/tensorflow aktif).
N_THREADS = int(os.environ.get('TA_THREADS', '2'))
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(_v, str(N_THREADS))

sys.path.insert(0, os.path.join('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
try:
    tf.config.threading.set_intra_op_parallelism_threads(N_THREADS)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    print('[PERHATIAN] TensorFlow sudah aktif di kernel ini, batas thread tidak berlaku. Restart kernel.')

from src.config import (OUT, HORIZONS, SETUPS, ML_MODELS, DL_MODELS, ALL_MODELS, LANG, PROTOCOL, ML_REV, QUICK,
                        SEQ_LEN, EPOCHS, PATIENCE, BASE_LR, BASE_WD, DL_BATCH, DL_UNITS)
from src import analysis as A, plots as P, stats as S, runners as R
from src.pipeline import SEED, load_mossa, free_memory
from src.runners import load_data, data_fingerprint

P.set_lang(LANG)
RES = os.path.join(OUT, 'results')
RUNS = os.path.join(RES, 'runs')
FIGS = os.path.join(OUT, 'figures')
MODEL_DIR = os.path.join(RES, 'models')
for _d in (RES, RUNS, FIGS, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

df, p, exo = load_data('..')
DATA_HASH = data_fingerprint(df)
print(f'Data: {len(p)} hari kerja, {p.index[0].date()} s.d. {p.index[-1].date()} | sidik jari {DATA_HASH}'
      + (' | MODE QUICK (hasil hanya uji kelancaran)' if QUICK else ''))


def simpan(fig, nama):
    P.save_fig(fig, nama, FIGS)
    plt.show()
    plt.close(fig)

from src.config import SHAP_SETUP, SHAP_H
DL_BASELINE = [m for m in DL_MODELS if m != 'dpformer']
FORCE_RERUN = False
print(f'Total run: {len(DL_BASELINE)} model x {len(HORIZONS)} horizon x {len(SETUPS)} setting = {len(DL_BASELINE) * len(HORIZONS) * len(SETUPS)}')

Data: 4325 hari kerja, 2010-01-05 s.d. 2026-08-03 | sidik jari 7aea8db80d44
Total run: 10 model x 5 horizon x 2 setting = 100


## Jalankan semua kombinasi

In [2]:
def run_fn(setup, model, h):
    wp = os.path.join(MODEL_DIR, f'{model}_{setup}_h{h}.weights.h5') if (setup == SHAP_SETUP and h == SHAP_H) else None
    return R.run_dl(p, exo if setup == 'multivariat' else None, setup, model, h, weights_path=wp, data_hash=DATA_HASH)


tasks = [(s, m, h) for s in SETUPS for m in DL_BASELINE for h in HORIZONS]
cnt = R.run_grid(tasks, run_fn, RUNS, data_hash=DATA_HASH, force=FORCE_RERUN)
assert cnt['gagal'] == 0, f"Ada run gagal: {cnt['daftar_gagal']}"

RUN  univariat/lstm/h1 ... WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.

OK MAE=13,255 (279.0s, 13 epoch, terbaik=3)
RUN  univariat/lstm/h5 ... OK MAE=30,116 (146.7s, 11 epoch, terbaik=1)
RUN  univariat/lstm/h10 ... OK MAE=42,735 (200.6s, 14 epoch, terbaik=4)
RUN  univariat/lstm/h20 ... OK MAE=62,671 (173.8s, 12 epoch, terbaik=2)
RUN  univariat/lstm/h30 ... OK MAE=82,953 (135.0s, 12 epoch, terbaik=2)
RUN  univariat/gru/h1 ... OK MAE=13,319 (44.9s, 12 epoch, terbaik=2)
RUN  univariat/gru/h5 ... OK MAE=30,291 (41.1s, 11 epoch, terbaik=1)
RUN  univariat/gru/h10 ... OK MAE=42,743 (69.1s, 13 epoch, terbaik=3)
RUN  univariat/gru/h20 ... OK MAE=62,941 (72.0s, 11 epoch, terbaik=1)
RUN  univariat/gru/h30 ... OK MAE=83,356 (46.4s, 11 epoch, terbaik=1)
RUN  univariat/bilstm/h1 ... OK MAE=13,356 (469.9s, 16 epoch, terbaik=6)
RUN  univari

## Cek cepat hasil

In [3]:
hasil = R.read_runs(RUNS, SETUPS, DL_BASELINE, HORIZONS, data_hash=DATA_HASH)
print(f'Total hasil tersimpan: {len(hasil)} / {len(DL_BASELINE) * len(HORIZONS) * len(SETUPS)}')
display(hasil.pivot_table(index='model', columns=['setup', 'horizon'], values='MAE').round(0))
print('\nEpoch terbaik (median per model & setup). Nilai 1–4 = model cepat menghafal train lalu validasi memburuk;')
print('nilai besar dengan loss ≈ model nol = konvergen pelan ke rata-rata (lihat notebook 08 dan 12).')
display(hasil.pivot_table(index='model', columns='setup', values='best_epoch', aggfunc='median'))

Total hasil tersimpan: 100 / 100


setup       multivariat                                     univariat  \
horizon              1        5        10       20       30        1    
model                                                                   
bigru           13544.0  30363.0  42702.0  62910.0  83951.0   13320.0   
bilstm          13508.0  30557.0  42220.0  63710.0  83649.0   13356.0   
dlinear         13556.0  30485.0  42057.0  62119.0  83139.0   13337.0   
gru             13489.0  30510.0  42444.0  63170.0  83407.0   13319.0   
gru_att         13232.0  29911.0  42117.0  64336.0  82639.0   13263.0   
lstm            13361.0  29934.0  42185.0  63340.0  84098.0   13255.0   
lstm_att        13270.0  30190.0  42579.0  62629.0  83099.0   13258.0   
nlinear         14225.0  31603.0  44596.0  63562.0  82004.0   13995.0   
patchtst        13525.0  30156.0  42389.0  63354.0  81287.0   13170.0   
transformer     13537.0  29495.0  42717.0  71807.0  83414.0   13213.0   

setup                                            
horizon           5        10       20       30  
model                                            
bigru        30329.0  42765.0  63114.0  82907.0  
bilstm       30306.0  42812.0  62801.0  83098.0  
dlinear      30217.0  42803.0  62396.0  82287.0  
gru          30291.0  42743.0  62941.0  83356.0  
gru_att      30132.0  42669.0  62376.0  82395.0  
lstm         30116.0  42735.0  62671.0  82953.0  
lstm_att     30074.0  42677.0  62313.0  82459.0  
nlinear      31197.0  44457.0  63507.0  81814.0  
patchtst     30008.0  42190.0  62407.0  83115.0  
transformer  29794.0  42188.0  63204.0  82188.0


Epoch terbaik (median per model & setup). Nilai 1–4 = model cepat menghafal train lalu validasi memburuk;
nilai besar dengan loss ≈ model nol = konvergen pelan ke rata-rata (lihat notebook 08 dan 12).


setup,multivariat,univariat
model,,
bigru,2.0,1.0
bilstm,2.0,2.0
dlinear,62.0,49.0
gru,2.0,1.0
gru_att,1.0,2.0
lstm,2.0,2.0
lstm_att,1.0,1.0
nlinear,69.0,67.0
patchtst,2.0,4.0
